In [ ]:
# ================================================================
# Notebook 04: ToN-IoT-rareclass-IDS Ablation Suite
#
# A. No MI, no PCA + OVR XGBoost + MITM KMeansSMOTE
# B. Correlation pruning only + OVR XGBoost + MITM KMeansSMOTE
# C. No MI, no PCA + OVR CatBoost + MITM KMeansSMOTE
# D. No MI, no PCA + LightGBM OVR + MITM KMeansSMOTE
# E. MITM sampling ratio sweep: 0.30, 0.50, 0.70, 0.90
# ================================================================

import pandas as pd
import numpy as np
from collections import Counter
import warnings
import time

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import KMeansSMOTE, SMOTE, BorderlineSMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False
    print("CatBoost is not installed. Install with: !pip install catboost")

print("\nNotebook 04: ToN-IoT-rareclass-IDS Ablation Suite\n")
start_time = time.time()

# ================================================================
# PARAMETERS
# ================================================================

RANDOM_STATE = 42

TRAIN_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_train_80.csv"
TEST_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_test_20.csv"

TARGET_COL = "type"
MITM_CLASS_NAME = "mitm"

THRESH_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

# ================================================================
# 1. LOAD DATA
# ================================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train 80%: {train_df.shape} | Test 20%: {test_df.shape}")

# ================================================================
# 2. CLEAN + CONSISTENT ENCODE
# ================================================================

def clean_and_encode_train_test(train_df, test_df):
    leak_cols = [
        "src_ip",
        "dst_ip",
        "http_uri",
        "ssl_subject",
        "ssl_issuer",
        "timestamp",
        "date",
        "time",
    ]

    train = train_df.drop(
        columns=[c for c in leak_cols if c in train_df.columns],
        errors="ignore",
    ).copy()

    test = test_df.drop(
        columns=[c for c in leak_cols if c in test_df.columns],
        errors="ignore",
    ).copy()

    train.replace([np.inf, -np.inf], np.nan, inplace=True)
    test.replace([np.inf, -np.inf], np.nan, inplace=True)

    for col in train.columns:
        if col in ["label", "type"]:
            continue

        if train[col].dtype == "object":
            mode_value = train[col].mode()[0]

            train[col] = train[col].fillna(mode_value).astype(str)
            test[col] = test[col].fillna(mode_value).astype(str)

            categories = pd.Index(train[col].unique())
            mapping = {cat: idx for idx, cat in enumerate(categories)}

            train[col] = train[col].map(mapping).astype(int)
            test[col] = test[col].map(mapping).fillna(-1).astype(int)

        else:
            mean_value = train[col].mean()
            train[col] = train[col].fillna(mean_value)
            test[col] = test[col].fillna(mean_value)

    return train, test


train_df, test_df = clean_and_encode_train_test(train_df, test_df)

# ================================================================
# 3. TRUE MULTI-CLASS TARGET SETUP
# ================================================================

X_train_full = train_df.drop(columns=["label", "type"], errors="ignore")
X_test_full = test_df.drop(columns=["label", "type"], errors="ignore")

label_encoder = LabelEncoder()

y_train_full = label_encoder.fit_transform(train_df[TARGET_COL].astype(str))
y_test_full = label_encoder.transform(test_df[TARGET_COL].astype(str))

class_names = list(label_encoder.classes_)
n_classes = len(class_names)

mitm_idx = class_names.index(MITM_CLASS_NAME)

print("\nClass mapping:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

print("\nClass distribution train:", Counter(y_train_full))
print("Class distribution test :", Counter(y_test_full))
print("\nMITM class index:", mitm_idx)

# ================================================================
# 4. TRAIN / CALIBRATION SPLIT
# ================================================================

sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

final_tr_idx, calib_idx = next(sss.split(X_train_full, y_train_full))

X_final_raw = X_train_full.iloc[final_tr_idx]
y_final = y_train_full[final_tr_idx]

X_calib_raw = X_train_full.iloc[calib_idx]
y_calib = y_train_full[calib_idx]

print("\nFinal train distribution:", Counter(y_final))
print("Calibration distribution:", Counter(y_calib))
print("Official test distribution:", Counter(y_test_full))

# ================================================================
# 5. HELPER FUNCTIONS
# ================================================================

def log_transform(X):
    return np.log1p(np.abs(X))


def build_features(
    X_train_raw,
    X_calib_raw,
    X_test_raw,
    feature_mode="none",
    corr_threshold=0.95,
):
    """
    feature_mode:
    - none: log transform + StandardScaler only
    - corr: log transform + StandardScaler + correlation pruning
    """

    X_train_log = log_transform(X_train_raw)
    X_calib_log = log_transform(X_calib_raw)
    X_test_log = log_transform(X_test_raw)

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_log)
    X_calib_scaled = scaler.transform(X_calib_log)
    X_test_scaled = scaler.transform(X_test_log)

    selected_idx = np.arange(X_train_scaled.shape[1])

    if feature_mode == "none":
        return X_train_scaled, X_calib_scaled, X_test_scaled, selected_idx

    if feature_mode == "corr":
        df_scaled = pd.DataFrame(X_train_scaled)

        corr_matrix = df_scaled.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        drop_cols = [
            col for col in upper.columns
            if any(upper[col] > corr_threshold)
        ]

        keep_cols = [col for col in df_scaled.columns if col not in drop_cols]
        selected_idx = np.array(keep_cols)

        X_train_selected = X_train_scaled[:, selected_idx]
        X_calib_selected = X_calib_scaled[:, selected_idx]
        X_test_selected = X_test_scaled[:, selected_idx]

        print(f"Correlation pruning kept {len(selected_idx)} features out of {df_scaled.shape[1]}")

        return X_train_selected, X_calib_selected, X_test_selected, selected_idx

    raise ValueError("feature_mode must be 'none' or 'corr'")


def mitm_kmeans_smote(X, y, mitm_ratio=0.70):
    counts = Counter(y)
    max_count = max(counts.values())

    target_mitm_count = int(max_count * mitm_ratio)

    if counts[mitm_idx] >= target_mitm_count:
        print("MITM already above target count. No oversampling applied.")
        return X, y

    sampling_strategy = {
        mitm_idx: target_mitm_count,
    }

    print("MITM sampling strategy:")
    print({class_names[k]: v for k, v in sampling_strategy.items()})

    samplers = [
        (
            "KMeansSMOTE",
            KMeansSMOTE(
                sampling_strategy=sampling_strategy,
                random_state=RANDOM_STATE,
                k_neighbors=3,
                cluster_balance_threshold=0.01,
            ),
        ),
        (
            "SMOTE",
            SMOTE(
                sampling_strategy=sampling_strategy,
                random_state=RANDOM_STATE,
                k_neighbors=3,
            ),
        ),
        (
            "BorderlineSMOTE",
            BorderlineSMOTE(
                sampling_strategy=sampling_strategy,
                random_state=RANDOM_STATE,
                k_neighbors=3,
            ),
        ),
    ]

    for sampler_name, sampler in samplers:
        try:
            X_res, y_res = sampler.fit_resample(X, y)
            print(f"{sampler_name} worked.")
            print("Before:", Counter(y))
            print("After :", Counter(y_res))
            return X_res, y_res
        except Exception as e:
            print(f"{sampler_name} failed: {str(e)[:150]}")

    print("All samplers failed. Continuing without resampling.")
    return X, y


def make_model(model_type):
    if model_type == "xgb":
        return XGBClassifier(
            n_estimators=350,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.2,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    if model_type == "lightgbm":
        return LGBMClassifier(
            n_estimators=350,
            max_depth=-1,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.2,
            objective="binary",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )

    if model_type == "catboost":
        if not CATBOOST_AVAILABLE:
            raise ImportError("CatBoost is not installed.")

        return CatBoostClassifier(
            iterations=350,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="F1",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )

    raise ValueError("model_type must be 'xgb', 'lightgbm', or 'catboost'")


def train_ovr_models(X, y, model_type):
    models = []

    original_counts = Counter(y)
    max_count = max(original_counts.values())

    for c in range(n_classes):
        print(f"Training OVR classifier for class {c}: {class_names[c]}")

        y_bin = (y == c).astype(int)

        classes_bin = np.array([0, 1])
        weights_bin = compute_class_weight(
            class_weight="balanced",
            classes=classes_bin,
            y=y_bin,
        )

        weight_map = {
            0: weights_bin[0],
            1: weights_bin[1],
        }

        sample_weight = np.array([weight_map[v] for v in y_bin])

        cls_count = original_counts[c]
        rare_multiplier = np.sqrt(max_count / cls_count)

        sample_weight[y_bin == 1] *= rare_multiplier

        if c == mitm_idx:
            sample_weight[y_bin == 1] *= 1.5

        model = make_model(model_type)
        model.fit(X, y_bin, sample_weight=sample_weight)

        models.append(model)

    return models


def predict_ovr_proba(models, X):
    probs = []

    for model in models:
        probs.append(model.predict_proba(X)[:, 1])

    return np.vstack(probs).T


def tune_thresholds(y_val, proba_val):
    thresholds = np.ones(n_classes) * 0.50

    for c in range(n_classes):
        y_true_bin = (y_val == c).astype(int)

        best_t = 0.50
        best_f1 = -1

        for t in THRESH_GRID:
            y_pred_bin = (proba_val[:, c] >= t).astype(int)
            score = f1_score(y_true_bin, y_pred_bin, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_t = t

        thresholds[c] = best_t

    return thresholds


def predict_with_thresholds(proba, thresholds):
    adjusted = proba / thresholds.reshape(1, -1)
    return np.argmax(adjusted, axis=1)


def mitm_metrics(y_true, y_pred):
    y_true_bin = (y_true == mitm_idx).astype(int)
    y_pred_bin = (y_pred == mitm_idx).astype(int)

    return {
        "mitm_precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "mitm_recall": recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "mitm_f1": f1_score(y_true_bin, y_pred_bin, zero_division=0),
    }


def evaluate_result(y_true, y_pred):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    metrics.update(mitm_metrics(y_true, y_pred))
    return metrics


def run_ablation(
    ablation_name,
    feature_mode="none",
    model_type="xgb",
    mitm_ratio=0.70,
    show_report=False,
):
    print("\n" + "=" * 100)
    print(f"RUNNING: {ablation_name}")
    print("=" * 100)
    print(f"Feature mode : {feature_mode}")
    print(f"Model type   : {model_type}")
    print(f"MITM ratio   : {mitm_ratio}")

    X_train_feat, X_calib_feat, X_test_feat, selected_idx = build_features(
        X_final_raw,
        X_calib_raw,
        X_test_full,
        feature_mode=feature_mode,
    )

    X_train_res, y_train_res = mitm_kmeans_smote(
        X_train_feat,
        y_final,
        mitm_ratio=mitm_ratio,
    )

    models = train_ovr_models(
        X_train_res,
        y_train_res,
        model_type=model_type,
    )

    proba_calib = predict_ovr_proba(models, X_calib_feat)
    thresholds = tune_thresholds(y_calib, proba_calib)

    print("\nThresholds:")
    print({class_names[i]: float(thresholds[i]) for i in range(n_classes)})

    proba_test = predict_ovr_proba(models, X_test_feat)
    y_test_pred = predict_with_thresholds(proba_test, thresholds)

    metrics = evaluate_result(y_test_full, y_test_pred)

    print("\nOfficial test metrics:")
    for k, v in metrics.items():
        print(f"{k:20s}: {v:.6f}")

    if show_report:
        print("\nPer-class report:")
        print(
            classification_report(
                y_test_full,
                y_test_pred,
                target_names=class_names,
                digits=4,
                zero_division=0,
            )
        )

        print("\nConfusion matrix:")
        print(confusion_matrix(y_test_full, y_test_pred))

    result = {
        "ablation": ablation_name,
        "feature_mode": feature_mode,
        "model_type": model_type,
        "mitm_ratio": mitm_ratio,
        "n_features": len(selected_idx),
    }

    result.update(metrics)

    return result, y_test_pred


# ================================================================
# 6. RUN ABLATIONS
# ================================================================

all_results = []
saved_predictions = {}

# ------------------------------------------------
# A. No MI, no PCA + OVR XGBoost + MITM KMeansSMOTE
# ------------------------------------------------

res_A, pred_A = run_ablation(
    ablation_name="A. No MI, no PCA + OVR XGBoost + MITM KMeansSMOTE",
    feature_mode="none",
    model_type="xgb",
    mitm_ratio=0.70,
    show_report=True,
)

all_results.append(res_A)
saved_predictions["A"] = pred_A

# ------------------------------------------------
# B. Correlation pruning only + OVR XGBoost + MITM KMeansSMOTE
# ------------------------------------------------

res_B, pred_B = run_ablation(
    ablation_name="B. Correlation pruning only + OVR XGBoost + MITM KMeansSMOTE",
    feature_mode="corr",
    model_type="xgb",
    mitm_ratio=0.70,
    show_report=True,
)

all_results.append(res_B)
saved_predictions["B"] = pred_B

# ------------------------------------------------
# C. No MI, no PCA + OVR CatBoost + MITM KMeansSMOTE
# ------------------------------------------------

if CATBOOST_AVAILABLE:
    res_C, pred_C = run_ablation(
        ablation_name="C. No MI, no PCA + OVR CatBoost + MITM KMeansSMOTE",
        feature_mode="none",
        model_type="catboost",
        mitm_ratio=0.70,
        show_report=True,
    )

    all_results.append(res_C)
    saved_predictions["C"] = pred_C
else:
    print("\nSkipping C because CatBoost is not installed.")

# ------------------------------------------------
# D. No MI, no PCA + LightGBM OVR + MITM KMeansSMOTE
# ------------------------------------------------

res_D, pred_D = run_ablation(
    ablation_name="D. No MI, no PCA + LightGBM OVR + MITM KMeansSMOTE",
    feature_mode="none",
    model_type="lightgbm",
    mitm_ratio=0.70,
    show_report=True,
)

all_results.append(res_D)
saved_predictions["D"] = pred_D

# ------------------------------------------------
# E. MITM sampling ratio sweep
# ------------------------------------------------

for ratio in [0.30, 0.50, 0.70, 0.90]:
    res_E, pred_E = run_ablation(
        ablation_name=f"E. No MI, no PCA + OVR XGBoost + MITM KMeansSMOTE ratio={ratio}",
        feature_mode="none",
        model_type="xgb",
        mitm_ratio=ratio,
        show_report=False,
    )

    all_results.append(res_E)
    saved_predictions[f"E_{ratio}"] = pred_E

# ================================================================
# 7. ABLATION SUMMARY
# ================================================================

results_df = pd.DataFrame(all_results)

results_df_sorted = results_df.sort_values(
    by=["weighted_f1", "macro_f1", "mitm_f1", "accuracy"],
    ascending=False,
)

print("\n" + "=" * 100)
print("FINAL ABLATION SUMMARY")
print("=" * 100)

display_cols = [
    "ablation",
    "accuracy",
    "weighted_f1",
    "macro_f1",
    "macro_recall",
    "balanced_accuracy",
    "mitm_precision",
    "mitm_recall",
    "mitm_f1",
    "n_features",
]

display(results_df_sorted[display_cols])

best_overall = results_df_sorted.iloc[0]

print("\nBest overall by weighted_f1 → macro_f1 → MITM F1:")
print(best_overall["ablation"])
print(best_overall[display_cols[1:]])

best_rare = results_df.sort_values(
    by=["mitm_f1", "macro_f1", "weighted_f1"],
    ascending=False,
).iloc[0]

print("\nBest rare-class result by MITM F1:")
print(best_rare["ablation"])
print(best_rare[display_cols[1:]])

# ================================================================
# 8. SAVE ABLATION RESULTS
# ================================================================

ablation_path = "/kaggle/working/table_2_ablation_results.csv"
results_df.to_csv(ablation_path, index=False)

ablation_rounded = results_df.copy()

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "mitm_precision",
    "mitm_recall",
    "mitm_f1",
]

for col in metric_cols:
    ablation_rounded[col] = ablation_rounded[col] * 100

ablation_rounded_path = "/kaggle/working/table_2_ablation_results_rounded.csv"
ablation_rounded.to_csv(ablation_rounded_path, index=False)

print("\nSaved ablation files:")
print(ablation_path)
print(ablation_rounded_path)

# ================================================================
# 9. GENERATE PAPER TABLES
# ================================================================

# Related work context table
related_work_context = pd.DataFrame([
    {
        "Model / Framework": "Reference paper reported result",
        "Dataset": "ToN-IoT",
        "Task": "Multi-class",
        "Accuracy (%)": 98.67,
        "F1-score (%)": 98.70,
        "Weighted F1 (%)": None,
        "Macro F1 (%)": None,
        "MITM Precision (%)": None,
        "MITM Recall (%)": None,
        "MITM F1 (%)": None,
        "Notes": "Reported headline result from related work",
    },
    {
        "Model / Framework": "ToN-IoT-rareclass-IDS: LightGBM OVR + MITM KMeansSMOTE",
        "Dataset": "ToN-IoT",
        "Task": "Multi-class",
        "Accuracy (%)": res_D["accuracy"] * 100,
        "F1-score (%)": res_D["weighted_f1"] * 100,
        "Weighted F1 (%)": res_D["weighted_f1"] * 100,
        "Macro F1 (%)": res_D["macro_f1"] * 100,
        "MITM Precision (%)": res_D["mitm_precision"] * 100,
        "MITM Recall (%)": res_D["mitm_recall"] * 100,
        "MITM F1 (%)": res_D["mitm_f1"] * 100,
        "Notes": "Independent rare-class-focused implementation",
    },
])

related_work_context_path = "/kaggle/working/table_1_related_work_context_rounded.csv"
related_work_context = related_work_context.round(2)
related_work_context.to_csv(related_work_context_path, index=False)

# Per-class report table for final model D
report_dict = classification_report(
    y_test_full,
    pred_D,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

per_class_table = pd.DataFrame(report_dict).transpose()
per_class_table = per_class_table.loc[class_names]

per_class_table = per_class_table.reset_index().rename(
    columns={
        "index": "Class",
        "precision": "Precision (%)",
        "recall": "Recall (%)",
        "f1-score": "F1-score (%)",
        "support": "Support",
    }
)

for col in ["Precision (%)", "Recall (%)", "F1-score (%)"]:
    per_class_table[col] = per_class_table[col] * 100

per_class_table["Support"] = per_class_table["Support"].astype(int)

per_class_table["Class Type"] = per_class_table["Class"].apply(
    lambda x: "Rare class" if x == "mitm" else "Common class"
)

per_class_table = per_class_table[
    [
        "Class",
        "Class Type",
        "Precision (%)",
        "Recall (%)",
        "F1-score (%)",
        "Support",
    ]
]

per_class_table = per_class_table.round(2)

per_class_table_path = "/kaggle/working/table_3_per_class_report_rounded.csv"
per_class_table.to_csv(per_class_table_path, index=False)

print("\nSaved paper table files:")
print(related_work_context_path)
print(ablation_rounded_path)
print(per_class_table_path)

print("\nTABLE 1: Related work context")
display(related_work_context)

print("\nTABLE 2: Ablation results")
display(ablation_rounded)

print("\nTABLE 3: Per-class report")
display(per_class_table)

print(f"\nNotebook 04 complete in {(time.time() - start_time) / 60:.2f} minutes.")

print("""
Notebook 04 conclusion:
This notebook compares the main ToN-IoT-rareclass-IDS ablation variants.
The best overall model is selected using weighted F1, macro-F1, MITM F1, and accuracy.
The final selected model is expected to be the LightGBM OVR variant with MITM-targeted KMeansSMOTE.
""")